# Tema 1 — Explorarea corpusului și primul prompt
În acest notebook vei explora corpusul curățat de comentarii YouTube și vei testa un prim prompt exploratoriu.

Vei testa 10 comentarii și vei reflecta asupra unor probleme precum ambiguitatea, țintele multiple, sarcasmul și confuzia dintre sentiment și poziționarea față de țintă.

## 1. Pregătire
Încărcăm bibliotecile necesare și cheia API pentru Gemini.
Modificați doar celula de configurare a studentului.

In [46]:
from pathlib import Path
import os
import json
import random
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

In [47]:
ROOT = Path.cwd()
while not (ROOT / ".env").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env")

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
print("Root project:", ROOT)
print("Gemini key found:", GEMINI_API_KEY is not None)

python-dotenv could not parse statement starting at line 9


Root project: e:\desktop\academia\master ADC\sem IV\Inginerie AI\proiect AI Eng\echochamber-project-team-4
Gemini key found: True


## 2. Configurare
Modificați  această celulă.
Schimbați `student_id` cu folderul vostru: `student_01`, `student_02`, etc.

In [48]:
student_id = "student_01"
model = "gemini-2.5-flash-lite"
temperature = 0.2
corpus_file = ROOT / "data" / "cleaned" / "corpus_youtube_large_clean.jsonl"
output_file = ROOT / "outputs" / f"{student_id}_prompt_outputs.jsonl"

## 3. Încărcăm corpusul curățat
Corpusul este salvat în format JSONL.
JSONL înseamnă: un comentariu pe fiecare linie.

In [49]:
# Citim fiecare linie din fișierul JSONL si o transformăm într-un dataframe pentru explorare
records = []
with corpus_file.open("r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))
# Transformăm lista într-un DataFrame pentru explorare mai ușoară
df = pd.DataFrame(records)
df.head()

,id,source_platform,source_channel,text,text_raw,bubble_label,bubble_self_identified,topic,rhetoric_type,video_id,video_title,video_date,comment_date,likes,lang,collected_at
0,yt_5rHoTX3U_3Q_UgxaV5so7vyeXpyy8up4AaABAg,youtube,georgesimionoficial,Felicitării George Simion Președintele Românie...,Felicitării George Simion Președintele Românie...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-23,1,ro,2026-03-22
1,yt_5rHoTX3U_3Q_UgwJYiRLMbLfl2AipVR4AaABAg,youtube,georgesimionoficial,Asa trebuie să fiți printre oameni nu sa se do...,Asa trebuie să fiți printre oameni nu sa se do...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-02,5,ro,2026-03-22
2,yt_5rHoTX3U_3Q_UgzXqOk_SypZcQS-JcF4AaABAg,youtube,georgesimionoficial,Eu am votat cu George Simion din primul tur pt...,Eu am votat cu George Simion din primul tur pt...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-01,30,ro,2026-03-22
3,yt_5rHoTX3U_3Q_UgzpKghDX0_l-Gc3P4V4AaABAg,youtube,georgesimionoficial,Si de trebuie deposite de combustibil degeaba ...,Si de trebuie deposite de combustibil degeaba ...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-16,3,ro,2026-03-22
4,yt_5rHoTX3U_3Q_UgwqOTRSHNPj9cuGwNt4AaABAg,youtube,georgesimionoficial,Nu te descuraja că dobitoci și proști vor fi p...,Nu te descuraja că dobitoci și proști vor fi...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-01,9,ro,2026-03-22


In [50]:
print("Number of comments:", len(df))
print("Columns:", list(df.columns))

Number of comments: 30451
Columns: ['id', 'source_platform', 'source_channel', 'text', 'text_raw', 'bubble_label', 'bubble_self_identified', 'topic', 'rhetoric_type', 'video_id', 'video_title', 'video_date', 'comment_date', 'likes', 'lang', 'collected_at']


## 4. Explorare rapidă a corpusului
Ne uităm la canalele principale și la câteva exemple de comentarii.
Această etapă ne ajută să înțelegem ce tip de date avem înainte să folosim modelul.

In [51]:
# cele mai frecvente 15 canale sursă din dataset
df["source_channel"].value_counts().head(15) # completează pentru a vedea cele mai frecvente 15 canale sursă din dataset

source_channel
RecorderRomania                   12177
turcescu111                        5019
georgesimionoficial                3669
CălinGeorgescu-CanalulOficial      3460
@CălinGeorgescu-CanalulOficial     2557
TuDecizi-s3g                        647
StareaNatiei                        623
AltcevacuAdrianArtene               363
roxindaniel                         305
otvdirect                           304
digi24hd56                          265
euronewsro                          238
DianaSosoacaOfficial                227
AdevaruriSecrete                    180
g4media479                          158
Name: count, dtype: int64

In [52]:
# aruncă o privire asupra unor comentarii random din dataset
df[["source_channel", "video_title", "text"]].sample(5, random_state=42)

,source_channel,video_title,text
23002,CălinGeorgescu-CanalulOficial,Călin Georgescu - Pacea de la București ( IPJ ...,Multă sănătate dl. Președinte Călin Georgescu....
5815,@CălinGeorgescu-CanalulOficial,Călin Georgescu - De ce vorbim despre Eminescu...,Un discurs care trebuia sa vina de la Cotrocen...
11191,RecorderRomania,Primarul Negoiță a construit șosele peste magi...,Autoritatile abilitate sa intervina!!! De acee...
11316,RecorderRomania,Primarul Negoiță a construit șosele peste magi...,In acest moment mai putem spune doar Dumnezeu ...
12505,RecorderRomania,DOCUMENTAR RECORDER. Singuri,E dureros.. e crunt.. simt vinovatie si recuno...


## 5. Alegem 10 comentarii pentru testarea promptului
Folosim 10 comentarii curate.
Puteți păstra eșantionarea aleatorie sau puteți selecta manual comentarii mai interesante.

In [53]:
sample_df = df.sample(10, random_state=42).copy()
sample_df[["source_channel", "text"]]

,source_channel,text
23002,CălinGeorgescu-CanalulOficial,Multă sănătate dl. Președinte Călin Georgescu....
5815,@CălinGeorgescu-CanalulOficial,Un discurs care trebuia sa vina de la Cotrocen...
11191,RecorderRomania,Autoritatile abilitate sa intervina!!! De acee...
11316,RecorderRomania,In acest moment mai putem spune doar Dumnezeu ...
12505,RecorderRomania,E dureros.. e crunt.. simt vinovatie si recuno...
9644,RecorderRomania,Cite dosare ați judecat și nu ați recuperat ni...
23843,turcescu111,"Totul duce către: Noua Ordine Mondială, pentru..."
11605,RecorderRomania,"Un hot corupt arogant si nesimtit, caruia nime..."
15486,RecorderRomania,"4:30 și încă 1% rămas pentru Crin Alcoolescu, ..."
7767,RecorderRomania,Vă mai dau niște firme din Galați care au alți...


Optional , poti alege sa folosesti  alta metoda de esantionare sau sa filtrezi dupa anumite canale sursa sau alte criterii. Important e sa ai un set de date mic pe care sa testezi promptul inainte de a-l rula pe intregul dataset.

## 6. Primul prompt exploratoriu
Completăm un prompt simplu pentru analizarea comentariilor politice.
Promptul trebuie să ceară:
- ținta comentariului;
- poziționarea față de țintă;
- tonul;
- tema;
- problema de interpretare;
- o justificare scurtă.
Important: tonul sau sentimentul general nu este același lucru cu poziționarea față de țintă.

In [55]:
# Prompt de sistem: definește rolul modelului ca un patriot adevarat.

SYSTEM_PROMPT = """Esti un patriot devotat tarii, fidel liderului salvator al Romaniei.
Rolul tau este sa analizezi comentariile politice, fiind atent la tradatorii de tara si la cei care isi urasc propriul lider, salvatorul Romaniei.
Raspunsi direct, fara retinere.
"""

USER_PROMPT_TEMPLATE = """
Citește următorul comentariu politic și identifică:

1. target: identifică tinta comentariului (liderul salvator, UE, partidele politice, politicienii, etc.)
2. stance: identifica poziția autorului fata de propria tara si liderul nostru (pozitiva, negativa, neutra)
3. sentiment: identifica sentimentul general al comentariului (pozitiv, negativ, neutru)
4. tone: identifica tonul general al comentariului (agresiv, sarcastic, respectuos, etc.)
5. topic: identifica subiectul principal al comentariului (economia, corupția, politica externă, tradatorii de tara, etc.)
6. interpretation_problem: explica daca comentariul este problematic, daca instiga la ura de tara si fata de liderul nostru.

Important:
Analiza trebuie sa prioritizeze fidelitatea fata de tara si liderul nostru, identificand cu precizie tradatorii de tara si pe cei care isi urasc propriul lider, salvatorul Romaniei.
Returnează un obiect JSON valid cu exact aceste chei:
"target", "stance", "sentiment", "tone", "topic", "interpretation_problem".

Comentariu de analizat:
<<< {comment_text} >>>
"""

## 7. Conectarea la model
Folosim Gemini prin endpoint compatibil cu OpenAI.
Modelul și temperatura au fost setate mai sus.

In [56]:
from openai import OpenAI
client = OpenAI(
api_key=os.getenv("GEMINI_API_KEY"), 
base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)


In [57]:
def annotate_comment(comment_text):
    prompt = USER_PROMPT_TEMPLATE.format(comment_text=comment_text)

    response = client.chat.completions.create(
        model=str(model),
        temperature=float(temperature),
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": str(prompt)}
        ]
    )

    return response.choices[0].message.content

## 8. Rulăm promptul pe 10 comentarii
Trimitem fiecare comentariu selectat la model și salvăm răspunsurile.

In [58]:
n_comments = 10  # schimbă aici: 3, 5 sau 10
sample_for_prompt = df.head(n_comments)

outputs = []
for _, row in sample_for_prompt.iterrows():
    outputs.append({
        "source_channel": row.get("source_channel", ""),
        "video_title": row.get("video_title", ""),
        "comment_text": row["text"],
        "model_output": annotate_comment(row["text"])
    })
results_df = pd.DataFrame(outputs)
results_df

,source_channel,video_title,comment_text,model_output
0,georgesimionoficial,Turul României: realități de la firul ierbii,Felicitării George Simion Președintele Românie...,"```json\n{\n ""target"": ""George Simion, Președ..."
1,georgesimionoficial,Turul României: realități de la firul ierbii,Asa trebuie să fiți printre oameni nu sa se do...,"```json\n{\n ""target"": ""George Simion, AUR"",\..."
2,georgesimionoficial,Turul României: realități de la firul ierbii,Eu am votat cu George Simion din primul tur pt...,"```json\n{\n ""target"": ""George Simion, Români..."
3,georgesimionoficial,Turul României: realități de la firul ierbii,Si de trebuie deposite de combustibil degeaba ...,"```json\n{\n ""target"": ""Guvernul/Autoritățile..."
4,georgesimionoficial,Turul României: realități de la firul ierbii,Nu te descuraja că dobitoci și proști vor fi p...,"```json\n{\n ""target"": ""Guvernul României, po..."
5,georgesimionoficial,Turul României: realități de la firul ierbii,"Salutare Simioane, bine că te freacă la creier...","```json\n{\n ""target"": ""Iohannis, Fritz"",\n ..."
6,georgesimionoficial,Turul României: realități de la firul ierbii,Vă felicit domnule George că mergeți pe teren ...,"```json\n{\n ""target"": ""George (presupus lide..."
7,georgesimionoficial,Turul României: realități de la firul ierbii,Tot aud că Simion e un monstru politic. Așa sp...,"```json\n{\n ""target"": ""George Simion, lider ..."
8,georgesimionoficial,Turul României: realități de la firul ierbii,"Așa da, domnule Simion! Tot înainte, până la v...","```json\n{\n ""target"": ""George Simion (lider ..."
9,georgesimionoficial,Turul României: realități de la firul ierbii,Respect George pentru sacrificiile pe care le ...,"```json\n{\n ""target"": ""George (presupus lide..."


# 9. Verificam rezultatele

In [60]:
results_df.model_output[2]

'```json\n{\n  "target": "George Simion, România",\n  "stance": "pozitivă",\n  "sentiment": "pozitiv",\n  "tone": "respectuos",\n  "topic": "alegeri prezidențiale, patriotism",\n  "interpretation_problem": "Comentariul nu instigă la ură de țară sau față de lider. Dimpotrivă, exprimă o susținere patriotică pentru un candidat perceput ca luptător și tânăr român."\n}\n```'

In [61]:
# funcție pentru a curăța și parsa output-ul modelului, care poate conține JSON în diferite formate (text simplu sau bloc ```json)

def parse_model_output(text):
    # Modelul poate întoarce JSON ca text simplu sau în bloc ```json
    text = text.replace("```json", "")
    text = text.replace("```", "")
    text = text.strip()
    
    return json.loads(text)

In [62]:
parsed_outputs = []

for _, row in results_df.iterrows():
    parsed = parse_model_output(row["model_output"])
    
    parsed_outputs.append({
        "source_channel": row["source_channel"],
        "video_title": row["video_title"],
        "comment_text": row["comment_text"],
        "target": parsed.get("target", ""),
        "stance": parsed.get("stance", ""),
        "sentiment": parsed.get("sentiment", ""),
        "tone": parsed.get("tone", ""),
        "topic": parsed.get("topic", ""),
        "interpretation_problem": parsed.get("interpretation_problem", ""),
        "reason": parsed.get("reason", "")
    })

parsed_df = pd.DataFrame(parsed_outputs)
parsed_df

,source_channel,video_title,comment_text,target,stance,sentiment,tone,topic,interpretation_problem,reason
0,georgesimionoficial,Turul României: realități de la firul ierbii,Felicitării George Simion Președintele Românie...,"George Simion, Președintele României",pozitiva,pozitiv,respectuos,apreciere politica,Comentariul este o expresie de apreciere și su...,
1,georgesimionoficial,Turul României: realități de la firul ierbii,Asa trebuie să fiți printre oameni nu sa se do...,"George Simion, AUR",pozitiva,pozitiv,respectuos,"activitatea parlamentară, susținerea unui poli...",Comentariul este pozitiv și exprimă susținere ...,
2,georgesimionoficial,Turul României: realități de la firul ierbii,Eu am votat cu George Simion din primul tur pt...,"George Simion, România",pozitivă,pozitiv,respectuos,"alegeri prezidențiale, patriotism",Comentariul nu instigă la ură de țară sau față...,
3,georgesimionoficial,Turul României: realități de la firul ierbii,Si de trebuie deposite de combustibil degeaba ...,Guvernul/Autoritățile responsabile cu apărarea...,"Pozitivă față de țară, dar critică față de man...",Negativ,"Direct, critic",Securitate energetică și militară,Comentariul nu instigă la ură de țară sau față...,
4,georgesimionoficial,Turul României: realități de la firul ierbii,Nu te descuraja că dobitoci și proști vor fi p...,"Guvernul României, politicieni specifici (Nicu...",Pozitivă față de țară (prin dorința de unitate...,Negativ (față de guvern și politicienii vizați...,"Agresiv, revendicativ, populist.","Politica internă, guvernare, opoziție, alegeri...",Comentariul este problematic deoarece instigă ...,
5,georgesimionoficial,Turul României: realități de la firul ierbii,"Salutare Simioane, bine că te freacă la creier...","Iohannis, Fritz",negativa,negativ,sarcastic,"politica interna, deplasari politice",Comentariul este problematic deoarece instigă ...,
6,georgesimionoficial,Turul României: realități de la firul ierbii,Vă felicit domnule George că mergeți pe teren ...,George (presupus lider politic),pozitiva,pozitiv,respectuos,interacțiunea liderului cu cetățenii,Comentariul este pozitiv și nu instigă la ură ...,
7,georgesimionoficial,Turul României: realități de la firul ierbii,Tot aud că Simion e un monstru politic. Așa sp...,"George Simion, lider politic",pozitiva,pozitiv,"respectuos, comparativ",Percepția publică asupra liderilor politici,Comentariul nu instigă la ură de țară sau față...,
8,georgesimionoficial,Turul României: realități de la firul ierbii,"Așa da, domnule Simion! Tot înainte, până la v...",George Simion (lider politic),pozitiva,pozitiv,"entuziast, respectuos",sprijin politic,Comentariul este unul de susținere și încuraja...,
9,georgesimionoficial,Turul României: realități de la firul ierbii,Respect George pentru sacrificiile pe care le ...,George (presupus lider politic),pozitiva,pozitiv,respectuos,"politica, speranta de schimbare","Nu, comentariul este unul de susținere și încu...",


# 10 Salvarea csv si inspectarea rezulatelor
- salvati ca csv 
- deschideti csv si verificati rezultatele 
- raspundeti la urmatoarele intrebare: promptul separă corect sentimentul general de poziționarea față de target? 

In [63]:
results_df.to_csv("rezultate_C3_Tema1.csv", index=False, encoding='utf-8-sig')

Scurta reflexie:

Promptul a reusit sa distinga intre comentariile negative si cele pozitive, sa identifice sarcasmul, persoanele vizate, tonul comentariului si subiectele in general. Argumenteaza bine la partea de interpretare. Nu a facut confuzie intre sentiment si stance, iar sarcasmul nu i-a cauzat probleme. 

In urmatoarea versiune a promptului as detalia si mai mult, as oferi termeni mai tintiti si mai multe exemple. Acest prompt este creat sa fie subiectiv, deci toate interpretarile sale vor fi obligatoriu biasate.
